In [ ]:
import pandas as pd
import requests
import numpy as np
from scipy.spatial import cKDTree

# ==========================================
# 1. LOAD EXCEL FILE
# ==========================================
print("1. Loading Excel file...")

df_stops = pd.read_excel("BusStops.xlsx")

df_stops['Coordinates'] = (
    df_stops['Coordinates']
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

df_stops[['Longitude', 'Latitude']] = (
    df_stops['Coordinates']
    .str.split(',', expand=True)
)

df_stops['Latitude'] = pd.to_numeric(df_stops['Latitude'].str.strip(), errors='coerce')
df_stops['Longitude'] = pd.to_numeric(df_stops['Longitude'].str.strip(), errors='coerce')

print(f"-> Successfully loaded {len(df_stops)} bus stops")

# ==========================================
# 2. BATCH WEATHER DATA (OPEN-METEO)
# ==========================================
print("2. Loading REAL weather data (Batched)...")

df_stops['Temperature'] = None
df_stops['Humidity'] = None

valid_mask = df_stops['Latitude'].notna() & df_stops['Longitude'].notna()
valid_indices = df_stops[valid_mask].index.tolist()

CHUNK_SIZE = 50 

for i in range(0, len(valid_indices), CHUNK_SIZE):
    chunk_indices = valid_indices[i:i+CHUNK_SIZE]
    chunk = df_stops.loc[chunk_indices]
    lats = ",".join(chunk['Latitude'].astype(str))
    lons = ",".join(chunk['Longitude'].astype(str))
    
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lats}"
        f"&longitude={lons}"
        f"&current=temperature_2m,relative_humidity_2m"
    )
    
    try:
        response = requests.get(url, timeout=15)
        data = response.json()

        if isinstance(data, dict) and 'current' in data:
            data = [data]
            
        for idx, res in zip(chunk_indices, data):
            if 'current' in res:
                df_stops.at[idx, 'Temperature'] = res['current'].get('temperature_2m')
                df_stops.at[idx, 'Humidity'] = res['current'].get('relative_humidity_2m')
                
    except Exception as e:
        print(f"Weather API error at chunk {i}:", e)

print("-> Weather data successfully loaded.")

# ==========================================
# 3. DOWNLOAD OSM DATA
# ==========================================
print("3. Downloading REAL OSM vegetation and benches...")

OSM_URL = "https://overpass-api.de/api/interpreter"
query = """
[out:json][timeout:40];
(
  node["natural"="tree"](47.45,21.50,47.60,21.72);
  node["amenity"="bench"](47.45,21.50,47.60,21.72);
);
out body;
"""

trees_data = []
benches_data = []

try:
    response = requests.post(OSM_URL, data={'data': query}, headers={'User-Agent': 'BusStopProject/1.0'}, timeout=60)
    data = response.json()
    elements = data.get("elements", [])
    print(f"-> Downloaded {len(elements)} OSM elements")

    for e in elements:
        lat, lon = e.get("lat"), e.get("lon")
        if lat is None or lon is None: continue
        
        tags = e.get("tags", {})
        if tags.get("natural") == "tree":
            trees_data.append([lat, lon])
        elif tags.get("amenity") == "bench":
            benches_data.append([lat, lon])

except Exception as e:
    print("OSM ERROR:", e)

print(f"-> Trees loaded: {len(trees_data)}")
print(f"-> Benches loaded: {len(benches_data)}")

# ==========================================
# 4. PROCESS BUS STOPS (SPATIAL KDTree)
# ==========================================
print("4. Processing spatial data with KDTree (Lightning fast)...")

RADIUS = 0.0015
df_stops['Vegetation'] = "No coordinates"
df_stops['Vegetation'] = df_stops['Vegetation'].astype(object)

df_stops['Spaces available'] = "No coordinates"
df_stops['Spaces available'] = df_stops['Spaces available'].astype(object)

if valid_mask.any():
    stop_coords = df_stops.loc[valid_mask, ['Latitude', 'Longitude']].values
    
    if trees_data:
        tree_tree = cKDTree(trees_data)
        tree_indices = tree_tree.query_ball_point(stop_coords, r=RADIUS, p=np.inf)
        tree_counts = [len(idx) for idx in tree_indices]
    else:
        tree_counts = [0] * len(stop_coords)
    if benches_data:
        bench_tree = cKDTree(benches_data)
        bench_indices = bench_tree.query_ball_point(stop_coords, r=RADIUS, p=np.inf)
        bench_counts = [len(idx) * 3 for idx in bench_indices] # *3 a helyek miatt
    else:
        bench_counts = [0] * len(stop_coords)

    df_stops.loc[valid_mask, 'Vegetation'] = tree_counts
    df_stops.loc[valid_mask, 'Spaces available'] = bench_counts
# ==========================================
# 5. GENERATE ISSUES
# ==========================================
print("5. Generating issue reports...")

problems = []
for _, row in df_stops.iterrows():
    issues = []
    if str(row.get('Covered', '')).strip().lower() == 'no': issues.append("No shelter")
    if str(row.get('Lightning', '')).strip().lower() == 'no': issues.append("No street lighting")
    if str(row.get('Wheelchair accessible', '')).strip().lower() == 'no': issues.append("Not wheelchair accessible")
    if str(row.get('Bus bay available', '')).strip().lower() == 'no': issues.append("No bus bay available")
    
    problems.append(", ".join(issues) if issues else "No reported issues")

df_stops['Problems'] = problems

# ==========================================
# 6. CLEANUP & 7. SAVE JSON
# ==========================================
df_final = df_stops.drop(columns=['Latitude', 'Longitude'])

output_file = "BusStop.json"
df_final.to_json(output_file, orient="records", force_ascii=False, indent=4)

print(f"\n🎉 Program finished successfully! Output file: {output_file}")

# ==========================================
# 8. PREVIEW & SEND TO SERVER
# ==========================================
print("\nFirst 5 rows:")
display(df_final[['Bus stop', 'Vegetation', 'Temperature', 'Humidity', 'Spaces available', 'Problems']].head())

adatok_json = df_final.fillna("").to_dict(orient='records')
url = "http://127.0.0.1:8000/api/upload"

print("-> Adatok küldése a FastAPI backendnek...")  
try:
    response = requests.post(url, json=adatok_json)
    print("-> Szerver (FastAPI) válasza:", response.json())
except Exception as e:
    print("-> Hiba történt a küldéskor. (Biztosan elindítottad a FastAPI backendet a háttérben?)\n Részletek:", e)

1. Loading Excel file...
-> Successfully loaded 687 bus stops
2. Loading REAL weather data (Batched)...
-> Weather data successfully loaded.
3. Downloading REAL OSM vegetation and benches...
-> Downloaded 1250 OSM elements
-> Trees loaded: 832
-> Benches loaded: 418
4. Processing spatial data with KDTree (Lightning fast)...
5. Generating issue reports...

🎉 Program finished successfully! Output file: BusStop.json

First 5 rows:


,Bus stop,Vegetation,Temperature,Humidity,Spaces available,Problems
0,Leiningen utca,0,22.9,40,0,"No shelter, No bus bay available"
1,Leiningen utca,0,22.9,40,0,No reported issues
2,Somlyai utca,0,23.0,39,0,No reported issues
3,Somlyai utca,0,23.0,39,0,No reported issues
4,Repülőtéri út,0,23.0,39,0,"No shelter, No bus bay available"


-> Adatok küldése a FastAPI backendnek...
-> Szerver (FastAPI) válasza: {'status': 'success', 'message': 'Adat sikeresen fogadva Simitől!'}
